# Big Data — Gastos eleitorais dos Prefeitos do Nordeste

Notebook independente do estudo de Presidente e Governador.

**Escopo:** eleições municipais de 2016, 2020 e 2024; somente candidatos ao cargo de **Prefeito**; nove estados do Nordeste (AL, BA, CE, MA, PB, PE, PI, RN e SE).

**Pergunta:** candidatos a prefeito que declararam maiores despesas tiveram maior frequência de eleição?

**Fonte:** Repositório de Dados Eleitorais e Portal de Dados Abertos do TSE.

**Importante:** o arquivo de prestação de contas de 2016 tem estrutura histórica diferente dos arquivos de 2020 e 2024. O código procura os arquivos oficiais e valida as colunas; se o layout antigo não permitir uma associação segura com a candidatura, o ano será sinalizado como erro — não será preenchido artificialmente com zero.

## 1. Bibliotecas e configurações

In [ ]:
!pip -q install pandas requests openpyxl

import io
import re
import time
import zipfile
import unicodedata
import pandas as pd
import requests

ANOS = [2016, 2020, 2024]
ESTADOS_NE = ["AL", "BA", "CE", "MA", "PB", "PE", "PI", "RN", "SE"]
CARGO_ANALISADO = "PREFEITO"
API_TSE = "https://dadosabertos.tse.jus.br/api/3/action/package_show"

print("Configurações carregadas.")
print("Estados:", ", ".join(ESTADOS_NE))
print("Cargo:", CARGO_ANALISADO)

## 2. Funções para localizar os conjuntos oficiais

In [ ]:
def normalizar(texto):
    texto = unicodedata.normalize("NFKD", str(texto).lower())
    texto = "".join(ch for ch in texto if not unicodedata.combining(ch))
    return re.sub(r"[^a-z0-9]+", " ", texto).strip()


def consultar_conjunto(slug):
    try:
        resposta = requests.get(API_TSE, params={"id": slug}, timeout=45)
        if resposta.status_code == 404:
            return None
        resposta.raise_for_status()
        dados = resposta.json()
        if dados.get("success"):
            return dados["result"].get("resources", [])
        return None
    except Exception as exc:
        print(f"Catálogo {slug}: {type(exc).__name__}: {exc}")
        return None


def primeiro_pacote(slugs):
    for slug in slugs:
        recursos = consultar_conjunto(slug)
        if recursos:
            print("Conjunto localizado:", slug)
            return recursos
    raise ValueError(f"Nenhum conjunto do TSE encontrado entre: {slugs}")


def escolher_recurso(recursos, termos):
    for recurso in recursos or []:
        nome = normalizar(recurso.get("name", ""))
        if all(t in nome for t in termos):
            return recurso
    return None


def localizar_recursos(ano):
    # O conjunto de 2020 possui o identificador 'candidatos-2020-subtemas';
    # procurar somente 'candidatos-2020' causa erro 404.
    slugs_candidatos = {
        2016: ["candidatos-2016"],
        2020: ["candidatos-2020-subtemas", "candidatos-2020"],
        2024: ["candidatos-2024", "candidatos-2024-subtemas"]
    }
    recursos_candidatos = primeiro_pacote(slugs_candidatos[ano])

    recurso_candidatos = (
        escolher_recurso(recursos_candidatos, ["candidatos"])
        or escolher_recurso(recursos_candidatos, ["consulta", "cand"])
    )
    if recurso_candidatos is None:
        raise ValueError(
            f"Arquivo de candidaturas não localizado em {ano}. "
            f"Recursos disponíveis: {[r.get('name') for r in recursos_candidatos[:20]]}"
        )

    if ano == 2016:
        # Nome correto do arquivo histórico de prestação de contas final.
        url_despesas = (
            "https://cdn.tse.jus.br/estatistica/sead/odsele/prestacao_contas/"
            "prestacao_contas_final_2016.zip"
        )
        nome_despesas = "prestacao_contas_final_2016.zip"
    else:
        # Arquivos nacionais de prestação de contas de candidatos de 2020 e 2024.
        url_despesas = (
            "https://cdn.tse.jus.br/estatistica/sead/odsele/prestacao_contas/"
            f"prestacao_de_contas_eleitorais_candidatos_{ano}.zip"
        )
        nome_despesas = f"prestacao_de_contas_eleitorais_candidatos_{ano}.zip"

    return {
        "candidatos": recurso_candidatos,
        "despesas": {"name": nome_despesas, "url": url_despesas}
    }


print("Funções de catálogo prontas.")

## 3. Download, leitura e identificação dos arquivos

In [ ]:
def baixar_conteudo(recurso, max_tentativas=3, intervalo=6):
    print("Baixando:", recurso.get("name"))
    erro_final = None
    for tentativa in range(1, max_tentativas + 1):
        try:
            resposta = requests.get(recurso["url"], timeout=300, stream=True)
            resposta.raise_for_status()
            buffer = io.BytesIO()
            for bloco in resposta.iter_content(chunk_size=1024 * 1024):
                if bloco:
                    buffer.write(bloco)
            return buffer.getvalue()
        except Exception as exc:
            erro_final = exc
            print(f"Tentativa {tentativa}/{max_tentativas}: {exc}")
            if tentativa < max_tentativas:
                time.sleep(intervalo)
    raise ConnectionError(f"Download falhou: {erro_final}")


def escolher_csv_candidatos(arquivo_zip):
    nomes = [n for n in arquivo_zip.namelist()
             if n.lower().endswith((".csv", ".txt"))]
    # Os nomes reais do TSE usam "consulta_cand"; não testar esse texto
    # depois de normalizar(), pois normalizar substitui "_" por espaço.
    opcoes = [
        n for n in nomes
        if re.search(r"consulta[_\\s-]*cand", n, flags=re.IGNORECASE)
    ]
    if not opcoes:
        opcoes = [
            n for n in nomes
            if "consulta" in normalizar(n)
            and "cand" in normalizar(n)
            and "despesa" not in normalizar(n)
        ]
    if not opcoes:
        raise ValueError(f"CSV de candidatos não encontrado. Arquivos: {nomes[:30]}")
    brasil = [n for n in opcoes if "brasil" in normalizar(n)]
    return sorted(brasil or opcoes, key=lambda n: ("brasil" not in normalizar(n), len(n)))[0]


def ler_csv_flexivel(conteudo, nome="arquivo"):
    ultimo_erro = None
    for encoding in ["latin1", "utf-8-sig", "utf-8"]:
        for separador in [";", ",", "\t"]:
            try:
                return pd.read_csv(
                    io.BytesIO(conteudo), sep=separador,
                    encoding=encoding, low_memory=False
                )
            except Exception as exc:
                ultimo_erro = exc
    raise ValueError(f"Não foi possível ler {nome}: {ultimo_erro}")


def baixar_e_ler_candidatos(recurso):
    conteudo = baixar_conteudo(recurso)
    if zipfile.is_zipfile(io.BytesIO(conteudo)):
        with zipfile.ZipFile(io.BytesIO(conteudo)) as z:
            nome = escolher_csv_candidatos(z)
            print("CSV de candidaturas escolhido:", nome)
            return ler_csv_flexivel(z.read(nome), nome)
    return ler_csv_flexivel(conteudo, recurso.get("name", "candidatos"))


def encontrar_coluna(df, alternativas, obrigatoria=True):
    for nome in alternativas:
        if nome in df.columns:
            return nome
    if obrigatoria:
        raise ValueError(
            f"Coluna não encontrada. Procurei {alternativas}. "
            f"Colunas disponíveis: {list(df.columns)}"
        )
    return None


def converter_moeda(serie):
    texto = serie.astype(str).str.strip().str.replace("R$", "", regex=False)
    texto = texto.str.replace(".", "", regex=False).str.replace(",", ".", regex=False)
    return pd.to_numeric(texto, errors="coerce").fillna(0)


print("Funções de leitura prontas.")

## 4. Selecionar somente Prefeitos dos nove estados

In [ ]:
def preparar_candidatos(df, ano):
    id_col = encontrar_coluna(df, ["SQ_CANDIDATO"])
    nome_col = encontrar_coluna(df, ["NM_URNA_CANDIDATO", "NM_CANDIDATO"])
    partido_col = encontrar_coluna(df, ["SG_PARTIDO"])
    cargo_col = encontrar_coluna(df, ["DS_CARGO"])
    uf_col = encontrar_coluna(df, ["SG_UF"])
    municipio_col = encontrar_coluna(df, ["NM_UE", "NM_MUNICIPIO"], obrigatoria=False)
    numero_col = encontrar_coluna(df, ["NR_CANDIDATO"], obrigatoria=False)

    df = df.copy()
    df[id_col] = df[id_col].astype(str).str.strip()
    df[cargo_col] = df[cargo_col].astype(str).str.upper().str.strip()
    df[uf_col] = df[uf_col].astype(str).str.upper().str.strip()

    filtro = df[uf_col].isin(ESTADOS_NE) & (df[cargo_col] == CARGO_ANALISADO)
    prefeitos = df.loc[filtro].copy()

    if prefeitos.empty:
        raise ValueError(
            f"Nenhum candidato a Prefeito encontrado em {ano}. "
            f"Estados identificados no arquivo: {sorted(df[uf_col].dropna().unique())}"
        )

    # Preserva o identificador TSE e as chaves auxiliares para os layouts antigos.
    saida = pd.DataFrame({
        "Ano": ano,
        "SQ_CANDIDATO": prefeitos[id_col].astype(str),
        "Candidato": prefeitos[nome_col],
        "Partido": prefeitos[partido_col],
        "Cargo": prefeitos[cargo_col],
        "Estado": prefeitos[uf_col],
        "Município": prefeitos[municipio_col] if municipio_col else "Não informado",
        "Número candidato": prefeitos[numero_col].astype(str) if numero_col else ""
    })

    situacao_col = encontrar_coluna(
        prefeitos, ["DS_SIT_TOT_TURNO", "DS_SITUACAO_TOT_TURNO"],
        obrigatoria=False
    )
    if situacao_col:
        situacao = prefeitos[situacao_col].fillna("").astype(str).str.upper()
        eleito = (
            situacao.str.contains("ELEITO")
            & ~situacao.str.contains("NÃO ELEITO")
            & ~situacao.str.contains("NAO ELEITO")
        )
        saida["Eleito"] = eleito.map({True: "Sim", False: "Não"})
    else:
        saida["Eleito"] = "Conferir resultado"

    return saida


print("Função de seleção de Prefeitos pronta.")

## 5. Processar despesas contratadas

Em 2020 e 2024, o ZIP possui arquivos separados por UF. O código processa explicitamente AL, BA, CE, MA, PB, PE, PI, RN e SE, evitando depender de um único arquivo consolidado que possa estar incompleto. Em 2016, usa o arquivo histórico `prestacao_contas_final_2016.zip` e valida o layout antes de somar.

In [ ]:
def nome_arquivo_despesa_por_uf(nomes, uf):
    norm = {n: normalizar(n) for n in nomes}
    candidatos = [
        n for n in nomes
        if "despesas contratadas" in norm[n]
        and "candidatos" in norm[n]
        and re.search(rf"(^| ){uf.lower()}($| )", norm[n])
        and "brasil" not in norm[n]
    ]
    return candidatos[0] if candidatos else None


def processar_csv_despesas(conteudo, ano, ids_validos, estado=None):
    # Cabeçalho para validar o arquivo antes de processar os dados.
    cab = ler_csv_flexivel(conteudo, "despesas")
    col_id = encontrar_coluna(cab, ["SQ_CANDIDATO"], obrigatoria=False)
    col_valor = encontrar_coluna(
        cab, ["VR_DESPESA_CONTRATADA", "VR_DESPESA", "VR_DESPESA_PAGA"],
        obrigatoria=False
    )
    if col_valor is None:
        raise ValueError(
            f"Arquivo de {ano} sem coluna de valor de despesa. "
            f"Colunas encontradas: {list(cab.columns)}"
        )
    if col_id is None:
        raise ValueError(
            f"Arquivo de {ano} sem SQ_CANDIDATO. "
            "O layout histórico exige associação por número, UF e município."
        )

    col_categoria = encontrar_coluna(
        cab, ["DS_TIPO_DESPESA", "DS_DESPESA", "DS_ORIGEM_DESPESA"],
        obrigatoria=False
    )
    dados = cab.copy()
    dados[col_id] = dados[col_id].astype(str).str.strip()
    dados = dados[dados[col_id].isin(ids_validos)].copy()
    if dados.empty:
        return {}, pd.DataFrame()

    dados[col_valor] = converter_moeda(dados[col_valor])
    if col_categoria:
        dados[col_categoria] = dados[col_categoria].fillna("Não informada").astype(str)
    else:
        dados["Categoria_aux"] = "Não informada"
        col_categoria = "Categoria_aux"

    totais = dados.groupby(col_id)[col_valor].sum().to_dict()
    categorias = dados.groupby([col_id, col_categoria], as_index=False)[col_valor].sum()
    categorias.columns = ["SQ_CANDIDATO", "Categoria de despesa", "Valor gasto (R$)"]
    categorias["Ano"] = ano
    if estado:
        print(f"{estado}: {len(dados):,} lançamentos de despesas lidos.")
    return {str(k): float(v) for k, v in totais.items()}, categorias


def processar_despesas_2020_2024(recurso, candidatos, ano):
    conteudo = baixar_conteudo(recurso)
    ids_por_estado = {
        uf: set(candidatos.loc[candidatos["Estado"] == uf, "SQ_CANDIDATO"].astype(str))
        for uf in ESTADOS_NE
    }
    totais_gerais = {}
    categorias_gerais = []

    with zipfile.ZipFile(io.BytesIO(conteudo)) as z:
        nomes = [n for n in z.namelist() if n.lower().endswith(".csv")]
        nomes_despesas = [n for n in nomes if "despesas contratadas" in normalizar(n)
                          and "candidatos" in normalizar(n)]
        if not nomes_despesas:
            raise ValueError(
                "O ZIP não contém arquivos de despesas contratadas de candidatos. "
                f"Arquivos encontrados: {nomes[:30]}"
            )

        for uf in ESTADOS_NE:
            ids = ids_por_estado[uf]
            if not ids:
                print(f"AVISO: nenhum prefeito selecionado em {uf}.")
                continue
            nome = nome_arquivo_despesa_por_uf(nomes_despesas, uf)
            if nome is None:
                # Fallback: tenta o arquivo BRASIL apenas se o arquivo da UF não existir.
                brasil = [n for n in nomes_despesas if "brasil" in normalizar(n)]
                if not brasil:
                    raise ValueError(f"Arquivo de despesas de {uf} não encontrado no ZIP.")
                nome = brasil[0]
                print(f"Arquivo específico de {uf} ausente; usando consolidado BRASIL.")
            print("Processando despesas:", nome)
            valores, categorias = processar_csv_despesas(
                z.read(nome), ano, ids, estado=uf
            )
            for cid, valor in valores.items():
                totais_gerais[cid] = totais_gerais.get(cid, 0.0) + valor
            if not categorias.empty:
                categorias_gerais.append(categorias)

    categorias_df = (
        pd.concat(categorias_gerais, ignore_index=True)
        if categorias_gerais else
        pd.DataFrame(columns=["SQ_CANDIDATO", "Categoria de despesa", "Valor gasto (R$)", "Ano"])
    )
    return totais_gerais, categorias_df


def processar_despesas_2016(recurso, candidatos):
    # O TSE publicou 2016 em layout histórico, separado do padrão 2020+.
    conteudo = baixar_conteudo(recurso)
    ids = set(candidatos["SQ_CANDIDATO"].astype(str))
    totais = {}
    categorias = []
    arquivos_lidos = []

    with zipfile.ZipFile(io.BytesIO(conteudo)) as z:
        membros = [n for n in z.namelist() if n.lower().endswith((".csv", ".txt"))]
        possiveis = [n for n in membros if "despesa" in normalizar(n)]
        if not possiveis:
            possiveis = membros

        for nome in possiveis:
            try:
                bruto = z.read(nome)
                amostra = pd.read_csv(io.BytesIO(bruto), sep=None, engine="python",
                                      encoding="latin1", nrows=5)
                col_id = encontrar_coluna(amostra, ["SQ_CANDIDATO"], obrigatoria=False)
                col_valor = encontrar_coluna(
                    amostra, ["VR_DESPESA_CONTRATADA", "VR_DESPESA", "VR_DESPESA_PAGA"],
                    obrigatoria=False
                )
                if col_id is None or col_valor is None:
                    continue

                df = ler_csv_flexivel(bruto, nome)
                df[col_id] = df[col_id].astype(str).str.strip()
                df = df[df[col_id].isin(ids)].copy()
                if df.empty:
                    continue
                df[col_valor] = converter_moeda(df[col_valor])
                cat_col = encontrar_coluna(
                    df, ["DS_TIPO_DESPESA", "DS_DESPESA", "DS_ORIGEM_DESPESA"],
                    obrigatoria=False
                )
                if cat_col is None:
                    df["Categoria_aux"] = "Não informada"
                    cat_col = "Categoria_aux"
                soma = df.groupby(col_id)[col_valor].sum()
                for cid, valor in soma.items():
                    totais[str(cid)] = totais.get(str(cid), 0.0) + float(valor)
                grupo = df.groupby([col_id, cat_col], as_index=False)[col_valor].sum()
                grupo.columns = ["SQ_CANDIDATO", "Categoria de despesa", "Valor gasto (R$)"]
                grupo["Ano"] = 2016
                categorias.append(grupo)
                arquivos_lidos.append(nome)
            except Exception:
                continue

    if not arquivos_lidos:
        raise ValueError(
            "O ZIP de 2016 foi baixado, mas não encontrei tabela com "
            "SQ_CANDIDATO e coluna de valor de despesa compatível. "
            "O layout histórico precisa de associação específica; nenhum gasto foi inventado."
        )

    print("Arquivos de despesas 2016 utilizados:", arquivos_lidos)
    return totais, (
        pd.concat(categorias, ignore_index=True) if categorias else pd.DataFrame()
    )


print("Funções de despesas prontas.")

## 6. Executar os três anos

In [ ]:
# Recria a leitura aqui para que a Seção 6 funcione mesmo se a Seção 3
# não tiver sido executada novamente após uma atualização do notebook.
def escolher_csv_candidatos(arquivo_zip):
    nomes = [
        n for n in arquivo_zip.namelist()
        if n.lower().endswith((".csv", ".txt"))
    ]
    opcoes = [
        n for n in nomes
        if re.search(r"consulta[_\\s-]*cand", n, flags=re.IGNORECASE)
    ]
    if not opcoes:
        raise ValueError(
            "Não encontrei arquivo consulta_cand. "
            f"Arquivos no ZIP: {nomes[:40]}"
        )
    brasil = [n for n in opcoes if "brasil" in normalizar(n)]
    escolhidos = brasil or opcoes
    escolhidos.sort(key=lambda n: (len(n), n))
    print("CSV de candidaturas selecionado:", escolhidos[0])
    return escolhidos[0]


def baixar_e_ler_candidatos(recurso):
    conteudo = baixar_conteudo(recurso)
    if zipfile.is_zipfile(io.BytesIO(conteudo)):
        with zipfile.ZipFile(io.BytesIO(conteudo)) as z:
            nome = escolher_csv_candidatos(z)
            return ler_csv_flexivel(z.read(nome), nome)
    return ler_csv_flexivel(conteudo, recurso.get("name", "candidatos"))


registros = []
categorias_por_ano = {}
erros = []
auditoria_estados = []

for ano in ANOS:
    print("\n" + "=" * 72)
    print("PROCESSANDO PREFEITOS:", ano)
    try:
        recursos = localizar_recursos(ano)
        base_candidatos = baixar_e_ler_candidatos(recursos["candidatos"])
        candidatos = preparar_candidatos(base_candidatos, ano)

        print("Candidatos a prefeito encontrados:", len(candidatos))
        print("Estados encontrados:", sorted(candidatos["Estado"].unique()))

        # Validação obrigatória dos nove estados: evita concluir o ano com cobertura parcial.
        estados_encontrados = set(candidatos["Estado"].unique())
        estados_faltantes = sorted(set(ESTADOS_NE) - estados_encontrados)
        if estados_faltantes:
            raise ValueError(
                f"Cobertura estadual incompleta em {ano}. "
                f"Estados ausentes: {estados_faltantes}. "
                f"Estados presentes: {sorted(estados_encontrados)}"
            )

        if ano == 2016:
            despesas, categorias = processar_despesas_2016(
                recursos["despesas"], candidatos
            )
        else:
            despesas, categorias = processar_despesas_2020_2024(
                recursos["despesas"], candidatos, ano
            )

        candidatos["Gasto total (R$)"] = (
            candidatos["SQ_CANDIDATO"].map(despesas).fillna(0)
        )
        candidatos["Possui despesa registrada"] = (
            candidatos["Gasto total (R$)"] > 0
        ).map({True: "Sim", False: "Não"})

        registros.append(candidatos)
        categorias_por_ano[ano] = categorias

        for uf in ESTADOS_NE:
            parte = candidatos[candidatos["Estado"] == uf]
            auditoria_estados.append({
                "Ano": ano,
                "Estado": uf,
                "Candidatos a prefeito": len(parte),
                "Candidatos com despesa": int((parte["Gasto total (R$)"] > 0).sum()),
                "Gasto total (R$)": parte["Gasto total (R$)"].sum()
            })

        print("Candidatos com despesas:", int((candidatos["Gasto total (R$)"] > 0).sum()))
        print("Gasto total do ano: R$", round(candidatos["Gasto total (R$)"].sum(), 2))
        print("ANO CONCLUÍDO COM OS NOVE ESTADOS.")
    except Exception as erro:
        erros.append({
            "Ano": ano,
            "Erro": f"{type(erro).__name__}: {erro}"
        })
        print("ERRO:", erros[-1]["Erro"])

print("\nRELATÓRIO DE ERROS")
display(pd.DataFrame(erros))
print("\nAUDITORIA DE COBERTURA POR ESTADO")
display(pd.DataFrame(auditoria_estados))

## 7. Tabela geral e Top 10

In [ ]:
colunas = [
    "Ano", "Estado", "Município", "SQ_CANDIDATO", "Candidato",
    "Partido", "Cargo", "Número candidato", "Gasto total (R$)",
    "Possui despesa registrada", "Eleito"
]
if registros:
    tabela_geral = pd.concat(registros, ignore_index=True)
    tabela_geral = tabela_geral[colunas].drop_duplicates(
        subset=["Ano", "SQ_CANDIDATO"]
    ).sort_values(
        ["Ano", "Estado", "Município", "Gasto total (R$)"],
        ascending=[True, True, True, False]
    ).reset_index(drop=True)
else:
    tabela_geral = pd.DataFrame(columns=colunas)

if tabela_geral.empty:
    print("Nenhum ano foi concluído.")
else:
    display(tabela_geral.style.format({"Gasto total (R$)": "R$ {:,.2f}"}))
    print("Total de candidatos:", len(tabela_geral))

if not tabela_geral.empty:
    top10 = (
        tabela_geral.sort_values("Gasto total (R$)", ascending=False)
        .groupby("Ano", group_keys=False).head(10).reset_index(drop=True)
    )
else:
    top10 = pd.DataFrame(columns=colunas)

print("10 maiores gastos por ano:")
display(top10.style.format({"Gasto total (R$)": "R$ {:,.2f}"}))

## 8. Comparar gastos e eleição

In [ ]:
if not tabela_geral.empty:
    resumo = (
        tabela_geral.groupby("Ano", as_index=False)
        .agg(
            Candidatos=("SQ_CANDIDATO", "nunique"),
            Eleitos=("Eleito", lambda s: (s == "Sim").sum()),
            Gasto total ("Gasto total (R$)", "sum"),
            Gasto médio=("Gasto total (R$)", "mean"),
            Gasto mediano=("Gasto total (R$)", "median")
        )
    )
    resumo["Percentual de eleitos"] = (
        resumo["Eleitos"] / resumo["Candidatos"] * 100
    )
    display(resumo.style.format({
        "Gasto total": "R$ {:,.2f}",
        "Gasto médio": "R$ {:,.2f}",
        "Gasto mediano": "R$ {:,.2f}",
        "Percentual de eleitos": "{:.2f}%"
    }))

    # Comparação por faixa de gasto dentro de cada ano.
    base = tabela_geral.copy()
    base["Faixa de gastos"] = base.groupby("Ano")["Gasto total (R$)"].transform(
        lambda s: pd.qcut(
            s.rank(method="first"), 4,
            labels=["Menor gasto", "2º grupo", "3º grupo", "Maior gasto"]
        )
    )
    comparacao_faixas = (
        base.groupby(["Ano", "Faixa de gastos"], observed=False)
        .agg(
            Candidatos=("SQ_CANDIDATO", "nunique"),
            Eleitos=("Eleito", lambda s: (s == "Sim").sum()),
            Gasto médio=("Gasto total (R$)", "mean")
        ).reset_index()
    )
    comparacao_faixas["Percentual de eleitos"] = (
        comparacao_faixas["Eleitos"] / comparacao_faixas["Candidatos"] * 100
    )
    display(comparacao_faixas.style.format({
        "Gasto médio": "R$ {:,.2f}",
        "Percentual de eleitos": "{:.2f}%"
    }))
else:
    resumo = pd.DataFrame()
    comparacao_faixas = pd.DataFrame()

## 9. Gastos por estado e município

In [ ]:
if not tabela_geral.empty:
    gastos_estado = (
        tabela_geral.groupby(["Ano", "Estado"], as_index=False)
        .agg(
            Candidatos=("SQ_CANDIDATO", "nunique"),
            Eleitos=("Eleito", lambda s: (s == "Sim").sum()),
            Gasto total=("Gasto total (R$)", "sum"),
            Gasto médio=("Gasto total (R$)", "mean")
        ).sort_values(["Ano", "Gasto total"], ascending=[True, False])
    )
    display(gastos_estado.style.format({
        "Gasto total": "R$ {:,.2f}", "Gasto médio": "R$ {:,.2f}"
    }))

    gastos_municipio = (
        tabela_geral.groupby(["Ano", "Estado", "Município"], as_index=False)
        .agg(
            Candidatos=("SQ_CANDIDATO", "nunique"),
            Eleitos=("Eleito", lambda s: (s == "Sim").sum()),
            Gasto total=("Gasto total (R$)", "sum"),
            Gasto médio=("Gasto total (R$)", "mean")
        ).sort_values(["Ano", "Gasto total"], ascending=[True, False])
    )
    display(gastos_municipio.head(100).style.format({
        "Gasto total": "R$ {:,.2f}", "Gasto médio": "R$ {:,.2f}"
    }))
else:
    gastos_estado = pd.DataFrame()
    gastos_municipio = pd.DataFrame()

## 10. Categorias de despesas dos maiores gastadores

In [ ]:
blocos = []
if not top10.empty:
    for ano in sorted(top10["Ano"].unique()):
        cat = categorias_por_ano.get(int(ano), pd.DataFrame())
        if cat is None or cat.empty:
            continue
        ids_top = set(top10.loc[top10["Ano"] == ano, "SQ_CANDIDATO"].astype(str))
        cat = cat[cat["SQ_CANDIDATO"].astype(str).isin(ids_top)].copy()
        nomes = top10.loc[top10["Ano"] == ano, ["SQ_CANDIDATO", "Candidato"]]
        cat = cat.merge(nomes, on="SQ_CANDIDATO", how="left")
        blocos.append(cat)

gastos_por_categoria = (
    pd.concat(blocos, ignore_index=True)
    if blocos else
    pd.DataFrame(columns=[
        "SQ_CANDIDATO", "Categoria de despesa", "Valor gasto (R$)", "Ano", "Candidato"
    ])
)
display(gastos_por_categoria)

## 11. Exportar as tabelas para Excel

In [ ]:
arquivo_excel = "analise_gastos_prefeitos_nordeste.xlsx"

with pd.ExcelWriter(arquivo_excel) as writer:
    tabela_geral.to_excel(writer, sheet_name="Todos os prefeitos", index=False)
    top10.to_excel(writer, sheet_name="Top 10 gastos", index=False)
    resumo.to_excel(writer, sheet_name="Resumo eleição", index=False)
    comparacao_faixas.to_excel(writer, sheet_name="Faixas de gastos", index=False)
    gastos_estado.to_excel(writer, sheet_name="Gastos por estado", index=False)
    gastos_municipio.to_excel(writer, sheet_name="Gastos por município", index=False)
    gastos_por_categoria.to_excel(writer, sheet_name="Categorias", index=False)
    pd.DataFrame(auditoria_estados).to_excel(writer, sheet_name="Auditoria estados", index=False)
    pd.DataFrame(erros).to_excel(writer, sheet_name="Erros", index=False)

print("Planilha criada:", arquivo_excel)
try:
    from google.colab import files
    files.download(arquivo_excel)
except ImportError:
    print("Arquivo salvo na pasta atual.")

## 12. Observações metodológicas

- O estudo considera somente Prefeitos nos nove estados nordestinos.
- O notebook interrompe o processamento de um ano se faltar qualquer estado. Assim, não apresenta uma análise parcial como se fosse regional completa.
- Para 2020 e 2024, despesas contratadas são processadas a partir dos arquivos estaduais dentro do ZIP nacional.
- Para 2016, o TSE disponibiliza um conjunto histórico separado. O código valida se o arquivo contém identificador de candidatura e valor de despesa compatíveis. Caso contrário, registra erro para evitar associar gastos ao candidato errado.
- A situação de eleito é lida da base de candidaturas quando a coluna de situação final está disponível.
- Despesas declaradas não provam que gastar mais causou a eleição. O estudo observa associação, não causalidade.
- Valores nominais de 2016, 2020 e 2024 não devem ser comparados sem correção da inflação.